# Module 34 — Transports: stdio over real pipes, and HTTP

**THE ONE IDEA:** the protocol is the same JSON either way. The **transport** only decides
how bytes move — and that choice decides **who can reach the server, and who holds the
credentials.**

| | `stdio` | Streamable HTTP |
|---|---|---|
| where the server runs | a **subprocess of the host** | anywhere on the network |
| framing | newline-delimited JSON on stdin/stdout | HTTP body, streamed |
| latency | microseconds | milliseconds |
| auth | inherited — **same trust boundary as the host** | must be enforced per request |
| multi-user | no, one session per process | yes |
| lifetime | dies with the host | independent |

A **real subprocess** is spawned below and a real handshake runs over real pipes.

> **Version note.** Your ladder README says a 2026-07-28 revision made MCP stateless and
> introduced MRTR, deprecating server-initiated `sampling`, `elicitation` and `roots`.
> **I could not verify that against a primary source**, so this module does not depend on
> it and does not describe MRTR. Confirm against the spec before teaching those details.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json, subprocess, sys, textwrap, pathlib, time

SERVER = pathlib.Path('_stdio_server.py')
SERVER.write_text(textwrap.dedent('''
    # A real MCP-shaped stdio server. Reads JSON-RPC lines from stdin and
    # writes JSON-RPC lines to stdout. That loop IS the entire transport.
    import sys, json
    TOOLS = {"search_policy": {"description": "Search bank policy.",
             "inputSchema": {"type": "object",
                             "properties": {"query": {"type": "string"}},
                             "required": ["query"]}}}
    POLICY = {"ltv": "Maximum LTV: 90% standard, 95% first-time buyers.",
              "erc": "ERC: 5% yr1, 4% yr2, 3% yr3, 2% yr4, 1% yr5."}
    for line in sys.stdin:                      # <- the transport, in one loop
        if not line.strip(): continue
        req = json.loads(line)
        m, p, i = req.get("method"), req.get("params", {}), req.get("id")
        if m == "initialize":
            r = {"protocolVersion": "2025-06-18", "capabilities": {"tools": {}},
                 "serverInfo": {"name": "stdio-policy", "version": "0.1"}}
        elif m == "tools/list":
            r = {"tools": [{"name": n, **t} for n, t in TOOLS.items()]}
        elif m == "tools/call":
            q = p["arguments"]["query"].lower()
            hit = next((v for k, v in POLICY.items() if k in q), "No policy matched.")
            r = {"content": [{"type": "text", "text": hit}], "isError": False}
        else:
            print(json.dumps({"jsonrpc": "2.0", "id": i,
                  "error": {"code": -32601, "message": m}}), flush=True); continue
        print(json.dumps({"jsonrpc": "2.0", "id": i, "result": r}), flush=True)
'''))
print("wrote", SERVER, f"({len(SERVER.read_text().splitlines())} lines)")

## Spawn it and talk over real pipes

In [ ]:
proc = subprocess.Popen([sys.executable, str(SERVER)],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                        text=True, bufsize=1)
print(f"subprocess pid={proc.pid} — a CHILD of this kernel, sharing its credentials")

def call(method, params=None, _id=[0]):
    _id[0] += 1
    proc.stdin.write(json.dumps({"jsonrpc": "2.0", "id": _id[0],
                                 "method": method, "params": params or {}}) + "\n")
    proc.stdin.flush()
    return json.loads(proc.stdout.readline())

t0 = time.time()
print("initialize  ->", call("initialize", {"protocolVersion": "2025-06-18"})["result"]["serverInfo"])
print("tools/list  ->", [t["name"] for t in call("tools/list")["result"]["tools"]])
out = call("tools/call", {"name": "search_policy", "arguments": {"query": "erc"}})
print("tools/call  ->", out["result"]["content"][0]["text"])
print(f"\n3 round trips over real pipes in {(time.time() - t0) * 1000:.1f} ms")

## Shut it down — the session IS the process

In [ ]:
proc.stdin.close(); proc.wait(timeout=5)
print("server exited, returncode =", proc.returncode)
print("\n^ no shutdown RPC was needed. Closing stdin ended the session, because")
print("  with stdio the process lifetime IS the session lifetime. Any state the")
print("  server held is now gone.")
SERVER.unlink(missing_ok=True)

## Why the choice matters

In [ ]:
print(f"{'concern':22} {'stdio':30} {'Streamable HTTP'}")
print("-" * 88)
for c, s, h in [
 ("who can reach it",  "only the host that spawned it", "anyone who can route to the URL"),
 ("credentials",       "INHERITED from the host env",   "must be checked PER REQUEST"),
 ("blast radius",      "one user's machine",            "every tenant on the server"),
 ("latency",           "microseconds",                  "milliseconds + network"),
 ("state",             "dies with the process",         "must be externalised"),
 ("scaling",           "one process per session",       "horizontal")]:
    print(f"{c:22} {s:30} {h}")

print("""
LESSON - the transport is a SECURITY decision wearing a performance costume.

  stdio  the server is a subprocess of the host, so it inherits the host's
         environment and credentials. That is why it is simple: there is no auth
         to build because there is no trust boundary to cross. It is also why a
         malicious stdio server is so dangerous - it already runs as you.
         Typosquatting a package name is the realistic attack.

  HTTP   the trust boundary is real, so authentication is NOT optional and must
         be per request. One server serves many tenants, so session isolation
         becomes your problem and the blast radius of a bug is everyone.

Practical rule: stdio for anything local and single-user - faster, less to get
wrong. Streamable HTTP the moment a server must be shared, outlive the host, or
sit behind an org boundary.

Note what module 33 said about state: with stdio the session IS the process.
Restart the host and everything the server remembered is gone.""")

---

**Next:** `35_mcp_in_an_agent_and_security.ipynb`